# Retrain channel-invariant InstanSeg on CPDMI at 0.325 µm/pixel

This notebook isolates the spatial-resolution experiment: it trains from scratch on the existing CPDMI Vectra + Zeiss training/validation records, keeps CODEX excluded, and does not train on TissueNet. It imports the packaged InstanSeg installation from the active `instanseg_training` Conda environment rather than any repository snapshot or fork.

The saved combined dataset may still contain TissueNet records; they are removed explicitly in memory before training. No raw-data loading or dataset rebuilding is required.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import shlex
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import instanseg

TRAINING_ROOT = Path(os.environ.get("INSTANSEG_TRAINING_ROOT", "/data1/lowes/ratnayn/Data/instanseg")).expanduser().resolve()
DATASET_PATH = TRAINING_ROOT / "datasets"
MODEL_PATH = TRAINING_ROOT / "models"
COMBINED_DATASET_FILE = DATASET_PATH / "segmentation_dataset.pth"

instanseg_import_path = Path(instanseg.__file__).resolve()
print(f"InstanSeg imported from: {instanseg_import_path}")
assert "site-packages/instanseg" in instanseg_import_path.as_posix(), (
    "Restart with the instanseg_training kernel; InstanSeg is not coming from its packaged Conda installation."
)
assert COMBINED_DATASET_FILE.exists(), f"Missing saved dataset: {COMBINED_DATASET_FILE}"
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(torch.cuda.current_device())
    print(f"GPU: {properties.name} ({properties.total_memory / 1024**3:.1f} GiB)")
else:
    raise RuntimeError("CUDA is not available in this notebook kernel.")


InstanSeg imported from: /data1/lowes/ratnayn/conda_envs/instanseg_training/lib/python3.11/site-packages/instanseg/__init__.py
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H200 NVL (139.8 GiB)


## Load the checkpoint and isolate CPDMI

`torch.load(..., weights_only=False)` is appropriate here because this is our own trusted dataset checkpoint containing arrays and metadata, not an untrusted model file.


In [2]:
Combined_Dataset = torch.load(
    COMBINED_DATASET_FILE,
    map_location="cpu",
    weights_only=False,
)

CPDMI_Dataset = {
    split: [item for item in items if item.get("parent_dataset") == "CPDMI_2023"]
    for split, items in Combined_Dataset.items()
}
del Combined_Dataset

assert CPDMI_Dataset["Train"], "No CPDMI training records found"
assert CPDMI_Dataset["Validation"], "No CPDMI validation records found"
assert all(
    item.get("parent_dataset") == "CPDMI_2023"
    for items in CPDMI_Dataset.values()
    for item in items
)
print({split: len(items) for split, items in CPDMI_Dataset.items()})


{'Train': 116, 'Validation': 30, 'Test': 0}


In [3]:
for split in ("Train", "Validation", "Test"):
    items = CPDMI_Dataset[split]
    if not items:
        print(f"{split}: empty (CODEX remains excluded)")
        continue
    print(
        f"{split}: items={len(items)}, "
        f"platforms={dict(Counter(item.get('platform') for item in items))}, "
        f"pixel_sizes={sorted({float(item['pixel_size']) for item in items})}, "
        f"channels={sorted({len(item.get('channel_names', [])) for item in items})}"
    )


Train: items=116, platforms={'Vectra': 101, 'Zeiss': 15}, pixel_sizes=[0.25, 0.325, 0.5], channels=[5, 7, 8]
Validation: items=30, platforms={'Vectra': 26, 'Zeiss': 4}, pixel_sizes=[0.25, 0.325, 0.5], channels=[5, 7, 8]
Test: empty (CODEX remains excluded)


## Resolution-dependent choices

The original model was trained at 0.5 µm/pixel. Moving to 0.325 µm/pixel makes every physical structure approximately `0.5 / 0.325 = 1.538×` larger in pixels and an object's pixel area approximately `2.37×` larger.

For the first controlled run, keep `tile_size=256` and `window_size=128`:

- This changes resolution while preserving the documented network/loss configuration.
- A 256-pixel tile spans 83.2 µm instead of 128 µm, but still contains multiple cells. The model is fully convolutional, so a larger tile does not enlarge its per-pixel architectural receptive field. The fixed-pixel receptive field also covers 35% less physical distance at 0.325 µm/pixel; this is a genuine resolution change that tile size cannot compensate for.
- A 128-pixel loss window spans 41.6 µm, which should still contain ordinary nuclei and whole cells.
- Training-time validation calls the shipped postprocessor with its 128-pixel default regardless of the trainer's `window_size`; changing only the training window would make loss and checkpoint selection inconsistent.
- Fixed 256-pixel crops keep GPU memory close to the working smoke test and reduce the chance of hitting the 50-instance-per-crop loss cap in dense fields.

A later physical-field-of-view ablation could use approximately `tile_size=384` and `window_size=192`. Pixel-based postprocessing parameters would also need reconsideration: the default peak separation scales from 4 to about 6 pixels, and the default 10-pixel minimum area scales to about 24 pixels. Those affect validation/inference and should be tuned together rather than changed only in training.

Other effects to monitor are crop-border truncation (cells occupy more of a 256-pixel tile), interpolation differences between native 0.25/0.325/0.5-µm records, and fragment/duplicate detections caused by leaving pixel-based postprocessing thresholds unchanged. The finer grid may preserve more boundary detail, but it also makes annotation edge noise and small nucleus/cell registration errors occupy more pixels. Fixed-size tiles keep feature-map memory broadly unchanged, but enlarging either tile or window increases memory approximately with pixel area. At inference, image pixel-size metadata must be correct so the model resamples to 0.325 µm/pixel rather than silently operating at the wrong biological scale.

Leave `mean_object_diameter=None`: all selected CPDMI records have physical pixel-size metadata, so the trainer can perform explicit resampling. In version 0.1.1, that metadata-driven branch overrides the generic random resize factor, so even the `heavy` preset resamples these records to the exact requested scale rather than adding scale jitter. That is desirable for this controlled baseline; scale robustness would require a later custom augmentation experiment. Learning rate, channel count, and intensity normalization do not require resolution-based scaling.


## Published-style CPDMI configuration

The channel-invariant paper reports 500 epochs, 1,000 batches of size 3, 256×256 crops, Adam, and a fixed learning rate of 0.001, plus the trainer's 10-epoch hot start. In the shipped trainer, `length_of_epoch` counts samples rather than batches, so 3,000 samples with batch 3 gives approximately 1,000 optimizer steps per epoch. The `heavy` preset is selected because the paper states that the additionally released public models used the expanded fluorescence augmentations; this is the closest preset supplied by version 0.1.1.


In [4]:
EXPERIMENT_NAME = "cpdmi_heavy_0325_tile256_window128_published_schedule_test"
BATCH_SIZE = 3
NUM_WORKERS = 8
NUM_EPOCHS = 500
LENGTH_OF_EPOCH = 3000
TILE_SIZE = 384
WINDOW_SIZE = 128
RNG_SEED = 42

training_kwargs = dict(
    source_dataset="[CPDMI_2023]",
    output_path=str(MODEL_PATH),
    experiment_str=EXPERIMENT_NAME,
    requested_pixel_size=0.325,
    target_segmentation="NC",
    channel_invariant=True,
    augmentation_type="heavy",
    weight=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_epochs=NUM_EPOCHS,
    length_of_epoch=LENGTH_OF_EPOCH,
    lr=0.001,
    tile_size=TILE_SIZE,
    window_size=WINDOW_SIZE,
    mean_object_diameter=None,
    hotstart_training=10,
    rng_seed=RNG_SEED,
)
training_kwargs


{'source_dataset': '[CPDMI_2023]',
 'output_path': '/data1/lowes/ratnayn/Data/instanseg/models',
 'experiment_str': 'cpdmi_heavy_0325_tile256_window128_published_schedule_test',
 'requested_pixel_size': 0.325,
 'target_segmentation': 'NC',
 'channel_invariant': True,
 'augmentation_type': 'heavy',
 'weight': False,
 'batch_size': 3,
 'num_workers': 8,
 'num_epochs': 500,
 'length_of_epoch': 3000,
 'lr': 0.001,
 'tile_size': 384,
 'window_size': 128,
 'mean_object_diameter': None,
 'hotstart_training': 10,
 'rng_seed': 42}

## One-epoch smoke test

This uses four optimizer steps and skips the hot start. Use a distinct experiment name so it cannot overwrite the full run.


In [5]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    from instanseg.scripts.train import instanseg_training

    smoke_kwargs = dict(training_kwargs)
    smoke_kwargs.update(
        experiment_str=f"{EXPERIMENT_NAME}_smoke",
        num_epochs=1,
        length_of_epoch=BATCH_SIZE * 4,
        hotstart_training=0,
    )
    torch.cuda.reset_peak_memory_stats()
    instanseg_training(segmentation_dataset=CPDMI_Dataset, **smoke_kwargs)
    print(f"Peak allocated CUDA memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
    print(f"Peak reserved CUDA memory: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GiB")


Saving results to /data1/lowes/ratnayn/Data/instanseg/models/cpdmi_heavy_0325_tile256_window128_published_schedule_test_smoke
Setting RNG seed to 42
Generating InstanSeg_UNet
<class 'list'> ['cpdmi_2023']
Datasets available in  Train
{('CPDMI_2023', 116)}
After filtering using:
{('CPDMI_2023', 116)}
Datasets available in  Validation
{('CPDMI_2023', 30)}
After filtering using:
{('CPDMI_2023', 30)}
Epoch: 0


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

/data1/lowes/ratnayn/conda_envs/instanseg_training/lib/python3.11/site-packages/instanseg/utils/AI_utils.py:96: RuntimeWarning: Mean of empty slice
  mean1_f1 = np.nanmean(f1_array, axis=0)


Saving model, best f1_score: 0.0
train_loss: 4.8552, test_loss: 2.511, training_time: 2, testing_time: 5, f1_score_nuclei: nan, f1_score_cells: 0, f1_score_joint: 0
Generating InstanSeg_UNet
Peak allocated CUDA memory: 9.81 GiB
Peak reserved CUDA memory: 11.78 GiB


## Full training

For a long run, the CLI command below is safer than keeping a notebook attached. The API cell is disabled by default.


In [6]:
RUN_FULL_TRAINING = True

if RUN_FULL_TRAINING:
    from instanseg.scripts.train import instanseg_training
    instanseg_training(segmentation_dataset=CPDMI_Dataset, **training_kwargs)
else:
    print("Full training disabled. Use the CLI command below for the long run.")


Saving results to /data1/lowes/ratnayn/Data/instanseg/models/cpdmi_heavy_0325_tile256_window128_published_schedule
Setting RNG seed to 42
Generating InstanSeg_UNet
<class 'list'> ['cpdmi_2023']
Datasets available in  Train
{('CPDMI_2023', 116)}
After filtering using:
{('CPDMI_2023', 116)}
Datasets available in  Validation
{('CPDMI_2023', 30)}
After filtering using:
{('CPDMI_2023', 30)}
Hotstart for 10 epochs with binary_xloss and dice_loss
Epoch: 0


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

/data1/lowes/ratnayn/conda_envs/instanseg_training/lib/python3.11/site-packages/instanseg/utils/pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  intersection = torch.sparse.mm(onehot1, onehot2.T).to_dense()


Saving model, best f1_score: 0.14862296546071177
train_loss: 1.3491, test_loss: 1.0105, training_time: 102, testing_time: 98, f1_score_nuclei: 0.13214, f1_score_cells: 0.16511, f1_score_joint: 0.14862
Epoch: 1


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
cli_command = [
    sys.executable, "-m", "instanseg.scripts.train",
    "--data_path", str(DATASET_PATH),
    "--dataset", "segmentation",
    "--source_dataset", "[CPDMI_2023]",
    "--output_path", str(MODEL_PATH),
    "--experiment_str", EXPERIMENT_NAME,
    "--requested_pixel_size", "0.325",
    "--target_segmentation", "NC",
    "--channel_invariant", "True",
    "--augmentation_type", "heavy",
    "--weight", "False",
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--num_epochs", str(NUM_EPOCHS),
    "--length_of_epoch", str(LENGTH_OF_EPOCH),
    "--lr", "0.001",
    "--tile_size", str(TILE_SIZE),
    "--window_size", str(WINDOW_SIZE),
    "--hotstart_training", "10",
    "--rng_seed", str(RNG_SEED),
]
print(shlex.join(cli_command))


## After training

Compare validation curves and segmentation overlays before changing spatial parameters. If the 128-pixel window truncates unusually large cells, run a separate 384/192 experiment rather than changing this baseline mid-run. Post-processing parameters should ultimately be optimized on the 0.325-µm validation data because `peak_distance`, `min_size`, and `window_size` are pixel-based.
